## Imports

In [1]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please u

In [2]:
from funcoes_escoragem import *

## Diretório

In [3]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [4]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [5]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 4 WEEK)
    AND DATE(data) <= CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,34470581453,2584582,2026-06-10,2026-06-10,1,Z,12056.000000000,BLEND3_3,2584582,"{""pessoas"":[{""nome"":""JOSE EDSON DE LIMA"",""docu...",...,34,1778785248777,"{""valor_aluguel"":1700,""matchmaking_on"":false,""...",2026-06-11 08:00:36+00:00,1781164833699753650,34470581453,C,2026-06-11 08:00:51.557000+00:00,2026-06-10 17:06:39+00:00,REPROVAR
1,3368069063,4128709,2026-06-09,2026-06-09,1,NI,4315.500000000,BLEND3_3,4128709,"{""pessoas"":[{""nome"":""ALISSON GODINHO SCHRAMM"",...",...,34,1778785248777,"{""valor_aluguel"":""3300"",""imobiliaria_id"":43214...",2026-06-09 18:00:43+00:00,1781028039728120380,03368069063,C,2026-06-09 18:09:02.285000+00:00,2026-06-09 13:25:01+00:00,DERIVAR
2,93307659049,4155180,2026-06-16,2026-06-16,1,C,4178.500000000,BLEND3_3,4155180,"{""pessoas"":[{""nome"":""AMELIA CRISTINA WAGNER"",""...",...,34,1778785248777,"{""valor_aluguel"":""1000"",""imobiliaria_id"":9730,...",2026-06-16 18:00:37+00:00,1781632834106194839,93307659049,C,2026-06-16 18:03:53.365000+00:00,2026-06-16 11:38:28+00:00,APROVAR
3,13372240650,4159155,2026-06-10,2026-06-11,1,NI,9727.000000000,BLEND3_3,4159155,"{""pessoas"":[{""nome"":""JEAN PIETRO DE ALMEIDA MO...",...,32,1778785248777,"{""valor_aluguel"":""2300"",""imobiliaria_id"":1476,...",2026-06-12 08:00:42+00:00,1781251237736005002,13372240650,B,2026-06-12 08:09:44.079000+00:00,2026-06-11 15:06:36+00:00,APROVAR
4,3525132760,4171471,2026-06-20,2026-06-20,1,B,2055.000000000,BLEND3_3,4171471,"{""pessoas"":[{""nome"":""EMERSON FERREIRA LEITE"",""...",...,32,1778785248777,"{""valor_aluguel"":""1750"",""imobiliaria_id"":15632...",2026-06-20 18:00:25+00:00,1781978422320031279,03525132760,B,2026-06-20 18:05:10.316000+00:00,2026-06-20 10:43:24+00:00,APROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113758,42960356810,4339323,2026-07-06,2026-07-06,1,E,2740.000000000,BLEND3_3,4339323,"{""pessoas"":[{""nome"":""NICOLLAS DE PAULA"",""docum...",...,32,1778785248777,"{""valor_aluguel"":1062.5,""matchmaking_on"":false...",2026-07-07 08:00:35+00:00,1783411232072338419,42960356810,B,2026-07-07 08:02:57.218000+00:00,2026-07-06 17:43:43+00:00,APROVAR
113759,11553075935,4339415,2026-07-06,2026-07-06,2,D,7055.500000000,HVA3,4339415,"{""pessoas"":[{""nome"":""ANDRESA ANGELA DA ROSA"",""...",...,34,1778785248777,"{""valor_aluguel"":4205,""matchmaking_on"":false,""...",2026-07-07 08:00:35+00:00,1783411232247529781,11553075935,C,2026-07-07 08:02:57.291000+00:00,2026-07-06 17:55:07+00:00,APROVAR
113760,26907615053,4339617,2026-07-06,2026-07-06,1,E,10343.500000000,BLEND3_3,4339617,"{""pessoas"":[{""nome"":""IZAR MARILENE FREITAS JAR...",...,32,1778785248777,"{""valor_aluguel"":775.57,""matchmaking_on"":false...",2026-07-07 08:00:35+00:00,1783411232655372195,26907615053,B,2026-07-07 08:02:57.540000+00:00,2026-07-06 18:57:58+00:00,APROVAR
113761,4694482025,4339703,2026-07-06,2026-07-06,1,E,753.500000000,BLEND3_3,4339703,"{""pessoas"":[{""nome"":""UILLIAN TROMBINI CORREA"",...",...,33,1778785248777,"{""valor_aluguel"":730.73,""matchmaking_on"":false...",2026-07-07 08:00:35+00:00,1783411232819336629,04694482025,E,2026-07-07 08:02:57.627000+00:00,2026-07-06 19:00:00+00:00,REPROVAR


In [6]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

In [7]:
credpago_df['model'].value_counts(dropna=False)

model
BLEND3_3                97324
BLEND_REGRESSAO_2026     8476
HVA3                     4496
BVS_CUSTOM               1721
BLEND_4                  1025
HVA4                      674
BVS_CUSTOM_V2              30
                           13
HFT1                        4
Name: count, dtype: int64

## Multiproponente

In [8]:
credpago_df['qtd_proponentes'].value_counts(dropna=False)

qtd_proponentes
1    109359
2      4059
3       312
4        31
5         1
6         1
Name: count, dtype: Int64

In [9]:
credpago_df['qtd_proponentes'].value_counts(normalize=True, dropna=False)

qtd_proponentes
1    0.961288
2    0.035679
3    0.002743
4    0.000272
5    0.000009
6    0.000009
Name: proportion, dtype: Float64

## Quebrar JSON - Retornado

In [10]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [11]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

contrato_df = (
    pd.json_normalize(payload_parsed, sep="_")
    .add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.
)

pessoa_df = pd.json_normalize(
    payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0]),
    sep="_",
).add_prefix("pessoa_")

In [12]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [13]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_df = (
    pd.json_normalize(request_parsed.dropna(), sep="_")
    .add_prefix("request_")
)

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [14]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [15]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


### CredPago - Imóvel e Histórico de Valor

In [16]:
# query = """
# WITH property_type AS (
#   SELECT
#     id AS contract_id,
#     CASE WHEN tp_imovel = 2 THEN 0 ELSE 1 END AS property_type,
#     CAST(id_cidade_ibge AS INT64) AS id_cidade_ibge,    --nova


#     TRIM(
#     REGEXP_REPLACE(
#         REGEXP_REPLACE(
#         UPPER(
#             REGEXP_REPLACE(
#             NORMALIZE(COALESCE(cidade, ''), NFD),
#             r'\pM', ''                      -- tira acento (marcas)
#             )
#         ),
#         r'[^A-Z0-9 ]', ' '                 -- tira lixo (agora já está tudo em maiúsculo)
#         ),
#         r'\s+', ' '                          -- colapsa espaços
#     )
#     ) AS contract_city_nm, --nova

#     CAST(id_imobiliaria AS INT64) AS agency_id,  --nova
#     DATE(criado_em) AS requested_at
#   FROM `loft-dl-fintech.bronze_credpago.imovel`
#   WHERE DATE(criado_em) >= DATE('2026-01-01')
#   QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY criado_em DESC) = 1
# ),

# rental_value AS (
#   SELECT
#     id_imovel AS contract_id,
#     CAST(vl_aluguel AS FLOAT64) AS rental_value,
#     DATE(data) AS requested_at
#   FROM `loft-dl-fintech.bronze_credpago.historico_valor`
#   WHERE DATE(data) >= DATE('2026-01-01')
#   QUALIFY ROW_NUMBER() OVER (PARTITION BY id_imovel ORDER BY data DESC) = 1
# )

# SELECT
#   COALESCE(p.contract_id, r.contract_id) AS contract_id,
#   GREATEST(p.requested_at, r.requested_at) AS requested_at,
#   p.property_type,
#   p.id_cidade_ibge,
#   p.contract_city_nm,
#   p.agency_id,
#   r.rental_value
# FROM property_type p
# FULL OUTER JOIN rental_value r
#   ON p.contract_id = r.contract_id;
# """

# credpago_imovel_df = pd.read_gbq(query, project_id=project_id)

In [17]:
# credpago_imovel_df.head()

### Merge 

In [18]:
# cp_final_df = pd.merge(credpago_clean, credpago_imovel_df.drop(columns=['requested_at']), on='contract_id', how='left')
# cp_final_df.head()

In [19]:
# cols_ = ["property_type", "rental_value", "id_cidade_ibge", "agency_id"]

# pct_nulls = (
#     cp_final_df[cols_].isna().mean().mul(100).sort_values(ascending=False)
# )
# pct_nulls

In [20]:
# pct_nulls_por_requested_at = (
#     cp_final_df.groupby("requested_at")[cols_]
#     .apply(lambda x: x.isna().mean().mul(100))
#     .sort_index()
# )

# pct_nulls_por_requested_at

## Escoragem Blend3

In [21]:
# rename_dict = {
#     "message_scores_BVS_CUSTOM": "SCRCRDPNMGRLPFLGBCLFCREDPGV1",#
#     "message_scores_HVA4": "SERASA_HVA4",
#     "pessoa_idade": "age",
#     "property_type": "property_type",  # peguei de fora da consulta realizada
#     "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
#     "rental_value": "rental_value",  # peguei de fora da consulta realizada
#     "message_rendaConsideradaContrato": "income",
#     "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
#     "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
#     "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
#     "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
# }

In [22]:
# vars_blend4 = ['SCRCRDPNMGRLPFLGBCLFCREDPGV1', 'SERASA_HVA4', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

# id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [23]:
# df_predict_blend3 = (cp_final_df.copy()).rename(columns=rename_dict)
# # df_predict = df_predict[id_vars+vars_blend4]
# df_predict_blend3["REGRA_BLEND_3"] = np.where(
#     df_predict_blend3["SCRCRDPNMGRLPFLGBCLFCREDPGV1"] >= 435,
#     "BLEND3",
#     "E_BVS",
# )

# df_predict_blend3["SCORE_BVS"] = df_predict_blend3["SCRCRDPNMGRLPFLGBCLFCREDPGV1"]
# df_predict_blend3["SCORE_SERASA"] = df_predict_blend3["SERASA_HVA4"]
# df_predict_blend3["RENDA"] = df_predict_blend3["income"]

# df_predict_blend3.head()

## Criação de Variáveis

In [24]:
# df_predict_blend3_vars = prepare_blend3_variables(df_predict_blend3)
# df_predict_blend3_escorada = predict_blend3(df_predict_blend3_vars)

## Rating

In [25]:
# bvs = pd.to_numeric(df_predict_blend3_escorada["SCORE_BVS"], errors="coerce")
# score = pd.to_numeric(df_predict_blend3_escorada["predict_blend3_2_to_score"], errors="coerce")

# conditions = [
#     bvs.between(800, 1000),                                    # 800 ≤ BVS ≤ 1000  → A
#     bvs.between(700, 799),                                     # 700 ≤ BVS < 800   → B
#     bvs.between(620, 699),                                     # 620 ≤ BVS < 700   → C
#     bvs.between(435, 619) & score.between(476, 1000),          # 435 ≤ BVS < 620, 476 ≤ pred ≤ 1000 → B
#     bvs.between(435, 619) & score.between(399, 475),           # 435 ≤ BVS < 620, 399 ≤ pred < 476  → C
#     bvs.between(435, 619) & score.between(387, 398),           # 435 ≤ BVS < 620, 387 ≤ pred < 399  → D
#     bvs.between(435, 619) & score.between(0, 386),             # 435 ≤ BVS < 620, 0 ≤ pred < 387    → E
#     bvs.between(0, 434),                                       # 0 ≤ BVS < 435    → E
# ]

# choices = ["A", "B", "C", "B", "C", "D", "E", "E"]

# df_predict_blend3_escorada["rating_manual_blend3"] = np.select(conditions, choices, default=None)

In [26]:
# bvs = pd.to_numeric(df_predict_blend3_escorada["SCORE_BVS"], errors="coerce")
# score = pd.to_numeric(df_predict_blend3_escorada["message_blendRegressaoPredict"], errors="coerce")

# conditions = [
#     bvs.between(800, 1000),                                    # 800 ≤ BVS ≤ 1000  → A
#     bvs.between(700, 799),                                     # 700 ≤ BVS < 800   → B
#     bvs.between(620, 699),                                     # 620 ≤ BVS < 700   → C
#     bvs.between(435, 619) & score.between(476, 1000),          # 435 ≤ BVS < 620, 476 ≤ pred ≤ 1000 → B
#     bvs.between(435, 619) & score.between(399, 475),           # 435 ≤ BVS < 620, 399 ≤ pred < 476  → C
#     bvs.between(435, 619) & score.between(387, 398),           # 435 ≤ BVS < 620, 387 ≤ pred < 399  → D
#     bvs.between(435, 619) & score.between(0, 386),             # 435 ≤ BVS < 620, 0 ≤ pred < 387    → E
#     bvs.between(0, 434),                                       # 0 ≤ BVS < 435    → E
# ]

# choices = ["A", "B", "C", "B", "C", "D", "E", "E"]

# df_predict_blend3_escorada["rating_json_blend3"] = np.select(conditions, choices, default=None)

## Salvar

In [27]:
# df_predict_blend3_escorada.to_csv(ANALYTICS_DIR/"df_predict_blend3.csv", index=False)

## Escoragem Blend4

In [28]:
cp_final_df = credpago_clean.replace({
    "request_imovelTipo": {"RESIDENCIAL": 1, "COMERCIAL": 0}, 
    "message_blend3Variables_hasPreviousContracts": {True: 1, False: 0},
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": {True: 1, False: 0}
}
)

In [29]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HFT1": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "request_imovelTipo": "property_type",
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "request_valorAluguel": "rental_value",
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [30]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [31]:
df_predict = (cp_final_df.copy()).rename(columns=rename_dict)
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] < 334,
    "E_BVS",
    "BLEND4",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,request_blend3Variables_cityPc4HistoryOver100Contracts,request_blend3Variables_realEstatePc4HistoryOver100Contracts,request_pessoa_principal_nome,qtde_pefin,valor_pefin_total,modalidades_pefin,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,34470581453,2584582,2026-06-10,2026-06-10,1,Z,BLEND3_3,5931060,34,34470581453,...,NaN,NaN,NaN,1.0,2476.0,FINANCIAMENT,BLEND4,NaN,NaN,12056.0
1,3368069063,4128709,2026-06-09,2026-06-09,1,NI,BLEND3_3,5918426,34,03368069063,...,NaN,NaN,NaN,2.0,147296.0,EMPRESTIMO,BLEND4,NaN,NaN,4315.5
2,93307659049,4155180,2026-06-16,2026-06-16,1,C,BLEND3_3,5953795,34,93307659049,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,4178.5
3,13372240650,4159155,2026-06-10,2026-06-11,1,NI,BLEND3_3,5935750,32,13372240650,...,NaN,NaN,NaN,1.0,12240.0,ALUGUEL,BLEND4,NaN,NaN,9727.0
4,3525132760,4171471,2026-06-20,2026-06-20,1,B,BLEND3_3,5978326,32,03525132760,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,2055.0


In [32]:
df_predict.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    113673
E_BVS         90
dtype: int64

In [33]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [34]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [35]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [36]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [37]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,34470581453,2584582,2026-06-10,2026-06-10,1,Z,BLEND3_3,5931060,34,34470581453,...,0.0,1.35,1.0,-0.638230,0.000000,0.000000,0.482286,518.0,A,A
1,3368069063,4128709,2026-06-09,2026-06-09,1,NI,BLEND3_3,5918426,34,03368069063,...,0.0,-0.10,3.0,0.739407,0.000000,0.719181,0.575410,425.0,C,D
2,93307659049,4155180,2026-06-16,2026-06-16,1,C,BLEND3_3,5953795,34,93307659049,...,0.0,0.70,0.0,-0.421070,0.000000,1.628419,0.557574,442.0,C,C
3,13372240650,4159155,2026-06-10,2026-06-11,1,NI,BLEND3_3,5935750,32,13372240650,...,0.0,-0.25,1.0,-0.427398,0.000000,-0.885736,0.477791,522.0,A,A
4,3525132760,4171471,2026-06-20,2026-06-20,1,B,BLEND3_3,5978326,32,03525132760,...,0.0,0.90,0.0,0.931352,0.000000,-1.389928,0.470645,529.0,A,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113758,42960356810,4339323,2026-07-06,2026-07-06,1,E,BLEND3_3,6055019,32,42960356810,...,0.0,-0.20,1.0,-0.093152,0.000000,0.412536,0.530982,469.0,B,A
113759,11553075935,4339415,2026-07-06,2026-07-06,2,D,HVA3,6055133,34,11553075935,...,0.0,-0.45,0.0,0.366774,0.000000,0.000000,0.524454,476.0,B,None
113760,26907615053,4339617,2026-07-06,2026-07-06,1,E,BLEND3_3,6055369,32,26907615053,...,0.0,1.70,0.0,-0.784077,-0.005731,-0.397626,0.512453,488.0,A,A
113761,4694482025,4339703,2026-07-06,2026-07-06,1,E,BLEND3_3,6055458,33,04694482025,...,0.0,-0.40,1.0,1.192442,-0.005731,-0.397626,0.567176,433.0,C,D


In [38]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [39]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    113673
E_BVS         90
dtype: int64

In [40]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           13
BLEND3_3                97324
BLEND_4                  1025
BLEND_REGRESSAO_2026     8476
BVS_CUSTOM               1721
BVS_CUSTOM_V2              30
HFT1                        4
HVA3                     4496
HVA4                      674
dtype: int64

In [41]:
cp_final_df.groupby("model", dropna=False).size()

model
                           13
BLEND3_3                97324
BLEND_4                  1025
BLEND_REGRESSAO_2026     8476
BVS_CUSTOM               1721
BVS_CUSTOM_V2              30
HFT1                        4
HVA3                     4496
HVA4                      674
dtype: int64

In [42]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    113673
E_BVS         90
dtype: int64

In [43]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)